# Sistema de recomendação

## Criando a matriz usuário e produto

Bibliotecas importantes

In [2]:
import pandas as pd
import numpy as np 
from sklearn.metrics.pairwise import cosine_similarity

reaproveito código da questão 6 porque é similar a premissa 

In [3]:
produtos = pd.read_csv('./1-lh_nautical_csv/products.csv')
variantes_produtos = pd.read_csv('./1-lh_nautical_csv/product_variants.csv')
pedidos = pd.read_csv('./1-lh_nautical_csv/orders.csv')
itens_pedido = pd.read_csv('./1-lh_nautical_csv/order_items.csv')

In [4]:
# primeiro preciso achar as colunas comum entre eles
print(produtos.columns)
print(variantes_produtos.columns)
print(pedidos.columns)
print(itens_pedido.columns)

Index(['id', 'name', 'description', 'brand_id', 'category_id', 'ncm_code',
       'unit_of_measure', 'is_active', 'created_at', 'updated_at'],
      dtype='str')
Index(['id', 'product_id', 'sku', 'barcode_ean', 'sale_price', 'cost_price',
       'weight_kg', 'icms_rate', 'ipi_rate', 'is_active', 'created_at',
       'updated_at'],
      dtype='str')
Index(['id', 'order_number', 'channel', 'customer_id', 'salesperson_id',
       'location_id', 'status', 'subtotal', 'discount_amount', 'total',
       'placed_at', 'created_at', 'updated_at'],
      dtype='str')
Index(['id', 'order_id', 'product_variant_id', 'quantity', 'unit_price',
       'icms_rate', 'ipi_rate', 'line_total'],
      dtype='str')


In [5]:
arq_1 = pd.merge(
    produtos,
    variantes_produtos,
    left_on='id',
    right_on='product_id',
    suffixes=('_produto', '_variante')
)

arq_2 = pd.merge(
    arq_1,
    itens_pedido,
    left_on='id_variante',
    right_on='product_variant_id'
)

arq_final = pd.merge(
    arq_2,
    pedidos,
    left_on='order_id',
    right_on='id'
)

Remoção de valores duplicados

In [6]:
interacoes = arq_final[['customer_id', 'id_produto']].drop_duplicates()

In [7]:
matriz_usuario_produto = pd.crosstab(
    interacoes['customer_id'],
    interacoes['id_produto']
)

In [8]:
print(matriz_usuario_produto)

id_produto   1    2    3    4    5    6    ...  495  496  497  498  499  500
customer_id                                ...                              
1              0    1    0    0    0    0  ...    0    0    0    0    0    0
2              0    0    0    0    0    0  ...    0    0    0    0    0    0
3              0    0    0    0    0    1  ...    0    0    0    0    0    0
4              0    0    0    0    1    0  ...    1    0    0    0    0    0
5              0    0    1    0    0    1  ...    0    0    1    0    0    0
...          ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...
1996           0    0    0    0    1    0  ...    0    0    0    0    0    0
1997           1    0    0    0    0    0  ...    0    0    0    0    0    0
1998           0    0    0    0    0    0  ...    0    0    1    0    0    0
1999           1    0    0    0    0    0  ...    0    0    0    0    0    0
2000           0    0    0    0    0    0  ...    0    1    1    0    1    0

Similaridade

Precisamos de sua transposta

In [11]:
matriz_produto_usuario = matriz_usuario_produto.T
similaridade = cosine_similarity(matriz_produto_usuario)
print(similaridade.shape)

(500, 500)


Transformamos em Data Frame para facilitar as operações

In [13]:
similaridade_data = pd.DataFrame(
    similaridade, 
    index = matriz_produto_usuario.index,
    columns = matriz_produto_usuario.index 
)

Agora pegamos o produto específico pedido: “Motor de Popa 1949”

In [14]:
item_de_referencia = produtos[
    produtos['name'] == "Motor de Popa 1949"
]
item_de_referencia

,id,name,description,brand_id,category_id,ncm_code,unit_of_measure,is_active,created_at,updated_at
179,180,Motor de Popa 1949,Motor de popa 4 tempos para embarcações de lazer,7,13,22146679,UN,True,2024-11-01 07:14:54,2024-11-10 04:23:05


In [15]:
# pegamos o id
id_motor = 180

# agora pegamos a similaridades do produto

similaridade_motor = similaridade_data.loc[id_motor]

Agora pegamos os 5 mais similares

In [ ]:
# antes removemos o próprio produto
similaridade_motor = similaridade_motor.drop(id_motor)
# top 5 produtos similares
top_5_similares = similaridade_motor.sort_values(
    ascending= False
).head(5)

top_5_similares

id_produto
389    0.256553
295    0.256239
75     0.255785
337    0.239332
55     0.237744
Name: 180, dtype: float64

# produto com maior similaridade 

In [19]:
maior_similaridade = top_5_similares.iloc[0]
print(maior_similaridade)

0.2565525786018399
